# Convertir archivos OpenSignals para Edge Impulse

Este notebook busca los archivos `.txt` dentro de `records_bitalino/label_*/raw`, convierte el canal ECG a CSV y guarda cada resultado en la carpeta `formateado` de su etiqueta. El nombre comienza con la etiqueta (por ejemplo, `label_01.SampleECG.csv`) para que Edge Impulse la infiera al subirlo.


## 1. Importar librerías y ubicar los datos


In [1]:
import glob
import importlib
import os

import pandas as pd
from IPython.display import display
from opensignalsreader import OpenSignalsReader
import utils_ei

# Recargar el módulo para recoger cambios hechos mientras el kernel estaba abierto.
importlib.reload(utils_ei)

# Funciona tanto si el notebook se abre desde esta carpeta como desde el workspace.
carpetas_candidatas = [
    os.path.join(os.getcwd(), "records_bitalino"),
    os.path.join(os.getcwd(), "7_2_multiple__files", "records_bitalino"),
    os.path.join(
        os.getcwd(), "7_edge_impulse", "7_2_multiple__files", "records_bitalino"
    ),
]
records_root = next(
    (os.path.abspath(path) for path in carpetas_candidatas if os.path.isdir(path)),
    None,
)

if records_root is None:
    raise FileNotFoundError(
        "No se encontró records_bitalino. Revisa la ubicación del notebook y de los datos."
    )

print(f"Carpeta de datos: {records_root}")

Carpeta de datos: e:\PROJECTS\ISB_COURSE\7_edge_impulse\7_2_multiple__files\records_bitalino


## 2. Encontrar los archivos crudos por etiqueta


In [2]:
label_dirs = sorted(
    path for path in glob.glob(os.path.join(records_root, "label_*"))
    if os.path.isdir(path)
)
if not label_dirs:
    raise FileNotFoundError(f"No se encontraron carpetas label_* en {records_root}")

raw_files_by_label = {}
for label_dir in label_dirs:
    label = os.path.basename(label_dir)
    raw_dir = os.path.join(label_dir, "raw")
    raw_files = sorted(glob.glob(os.path.join(raw_dir, "*.txt")))
    if not raw_files:
        raise FileNotFoundError(f"No hay archivos .txt en {raw_dir}")

    raw_files_by_label[label] = raw_files
    print(f"{label}: {len(raw_files)} archivos .txt")

label_01: 20 archivos .txt
label_02: 20 archivos .txt


## 3. Convertir cada archivo y guardarlo en `formateado`


In [3]:
sensor = "ECG"
units = "mV"
converted_files = []

for label, raw_files in raw_files_by_label.items():
    formatted_dir = os.path.join(records_root, label, "formateado")
    os.makedirs(formatted_dir, exist_ok=True)

    for raw_file in raw_files:
        acquisition = OpenSignalsReader(raw_file)
        ecg_signal = acquisition.signal(sensor)
        sample_name = os.path.splitext(os.path.basename(raw_file))[0]

        # Edge Impulse infiere la etiqueta del texto anterior al primer punto.
        output_name = f"{label}.{sample_name}.csv"
        
        output_file = utils_ei.opensignals_txt_to_clean_csv(
            ecg_signal,
            output_dir=formatted_dir,
            sensor=sensor,
            sampling_rate=float(acquisition.sampling_rate),
            units=units,
            filename=output_name,
        )

        converted_files.append({
            "label": label,
            "archivo_crudo": os.path.basename(raw_file),
            "archivo_csv": os.path.basename(output_file),
            "muestras": len(ecg_signal),
            "frecuencia_hz": float(acquisition.sampling_rate),
        })

    print(f"{label}: {len(raw_files)} CSV guardados en {formatted_dir}")

label_01: 20 CSV guardados en e:\PROJECTS\ISB_COURSE\7_edge_impulse\7_2_multiple__files\records_bitalino\label_01\formateado
label_02: 20 CSV guardados en e:\PROJECTS\ISB_COURSE\7_edge_impulse\7_2_multiple__files\records_bitalino\label_02\formateado


## 4. Revisar los archivos generados


In [4]:
conversion_summary = pd.DataFrame(converted_files)
display(conversion_summary)
print(f"Total convertido: {len(converted_files)} archivos")

,label,archivo_crudo,archivo_csv,muestras,frecuencia_hz
0,label_01,SampleECG (1) - Copy.txt,label_01.SampleECG (1) - Copy.csv,22350,1000.0
1,label_01,SampleECG (1).txt,label_01.SampleECG (1).csv,22350,1000.0
2,label_01,SampleECG (2) - Copy.txt,label_01.SampleECG (2) - Copy.csv,22350,1000.0
3,label_01,SampleECG (3) - Copy.txt,label_01.SampleECG (3) - Copy.csv,22350,1000.0
4,label_01,SampleECG (4) - Copy.txt,label_01.SampleECG (4) - Copy.csv,22350,1000.0
5,label_01,SampleECG (5) - Copy.txt,label_01.SampleECG (5) - Copy.csv,22350,1000.0
6,label_01,SampleECG (6) - Copy.txt,label_01.SampleECG (6) - Copy.csv,22350,1000.0
7,label_01,SampleECG (7) - Copy.txt,label_01.SampleECG (7) - Copy.csv,22350,1000.0
8,label_01,SampleECG (8) - Copy.txt,label_01.SampleECG (8) - Copy.csv,22350,1000.0
9,label_01,SampleECG (9) - Copy.txt,label_01.SampleECG (9) - Copy.csv,22350,1000.0


Total convertido: 40 archivos
